In [ ]:
# table 1 

# ============================================================
# IPV+TBI, IPV alone, Overall IPV,
# All CAPriCORN, All CAPriCORN ED
#
# Statistical comparison:
# IPV+TBI vs IPV alone
# ============================================================

import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, fisher_exact, mannwhitneyu


# ============================================================
# 1. Load row-level datasets
# ============================================================

ipv = pd.read_csv("data/ipv_table_1.csv")
cap = pd.read_csv("data/all_cap_table_1.csv")
cap_payer = pd.read_csv("data/capricorn_payer.csv")


# ============================================================
# 2. Clean cohort labels
# ============================================================

ipv["cohort_label"] = (
    ipv["cohort_label"]
    .astype(str)
    .str.strip()
    .replace({
        "IPV only": "IPV alone"
    })
)

cap["cohort_label"] = (
    cap["cohort_label"]
    .astype(str)
    .str.strip()
)

cap_payer["cohort_label"] = (
    cap_payer["cohort_label"]
    .astype(str)
    .str.strip()
)


# ============================================================
# 3. Add payer information to CAPriCORN dataframe
# ============================================================

cap = cap.merge(
    cap_payer[
        ["PATID", "cohort_label", "payer_cat"]
    ],
    on=["PATID", "cohort_label"],
    how="left"
)


# ============================================================
# 4. Create Overall IPV cohort
# ============================================================

overall_ipv = ipv[
    ipv["cohort_label"].isin(
        ["IPV+TBI", "IPV alone"]
    )
].copy()

overall_ipv["cohort_label"] = "Overall IPV"


# ============================================================
# 5. Combine all five groups
# ============================================================

df = pd.concat(
    [
        ipv,
        overall_ipv,
        cap
    ],
    ignore_index=True
)


# ============================================================
# 6. Cohort order
# ============================================================

cohort_order = [
    "IPV+TBI",
    "IPV alone",
    "Overall IPV",
    "All CAPriCORN",
    "All CAPriCORN ED"
]


# Check Ns
print("\nCohort sizes:")

print(
    df["cohort_label"]
    .value_counts()
    .reindex(cohort_order)
)


# ============================================================
# 7. Recode demographic variables
# ============================================================

sex_map = {
    "F": "Female",
    "M": "Male",
    "NI": "Unknown",
    "UN": "Unknown",
    "NULL": "Unknown"
}

race_map = {
    "02": "Asian",
    "03": "African American",
    "05": "White",
    "01": "Other",
    "04": "Other",
    "06": "Other",
    "07": "Unknown",
    "OT": "Other",
    "NI": "Unknown",
    "UN": "Unknown",
    "NULL": "Unknown"
}

eth_map = {
    "Y": "Hispanic",
    "N": "Non-Hispanic",
    "NI": "Unknown",
    "UN": "Unknown",
    "NULL": "Unknown",
    "R": "Unknown",
    "OT": "Other"
}

orientation_map = {
    "AS": "LGBTQ+",
    "BI": "LGBTQ+",
    "GA": "LGBTQ+",
    "LE": "LGBTQ+",
    "HO": "LGBTQ+",
    "QU": "LGBTQ+",
    "QS": "LGBTQ+",
    "ST": "Straight",
    "SE": "Other",
    "DC": "Unknown",
    "NI": "Unknown",
    "UN": "Unknown",
    "NULL": "Unknown"
}

lang_map = {
    "ENG": "English",
    "SPA": "Spanish",
    "NI": "Unknown",
    "UN": "Unknown"
}

payer_map = {
    "medicaid": "Medicaid",
    "medicare": "Medicare",
    "private": "Private",
    "other/unknown": "Other or unknown"
}


# ============================================================
# 8. Apply recoding
# ============================================================

df["SEX"] = (
    df["SEX"]
    .astype(str)
    .str.strip()
    .map(sex_map)
    .fillna("Unknown")
)

df["RACE"] = (
    df["RACE"]
    .astype(str)
    .str.strip()
    .map(race_map)
    .fillna("Other")
)

df["HISPANIC"] = (
    df["HISPANIC"]
    .astype(str)
    .str.strip()
    .map(eth_map)
    .fillna("Unknown")
)

df["SEXUAL_ORIENTATION"] = (
    df["SEXUAL_ORIENTATION"]
    .astype(str)
    .str.strip()
    .map(orientation_map)
    .fillna("Unknown")
)

df["PAT_PREF_LANGUAGE_SPOKEN"] = (
    df["PAT_PREF_LANGUAGE_SPOKEN"]
    .astype(str)
    .str.strip()
    .map(lang_map)
    .fillna("Other")
)

df["payer_cat"] = (
    df["payer_cat"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map(payer_map)
    .fillna("Other or unknown")
)

df["age_at_index"] = pd.to_numeric(
    df["age_at_index"],
    errors="coerce"
)


# ============================================================
# 9. Create age groups
# ============================================================

df["Age group"] = pd.cut(
    df["age_at_index"],
    bins=[
        18,
        35,
        45,
        55,
        65,
        np.inf
    ],
    right=False,
    labels=[
        "18-34",
        "35-44",
        "45-54",
        "55-64",
        "65+"
    ]
)


# ============================================================
# 10. Helper functions
# ============================================================

def format_n_pct(n, denominator):
    """
    Format categorical variables as N (%).
    """

    if denominator == 0:
        return "0 (0.0%)"

    return (
        f"{int(n):,} "
        f"({100 * n / denominator:.1f}%)"
    )


def p_fmt(p):
    """
    Format p-values for manuscript.
    """

    if pd.isna(p):
        return ""

    if p < 0.001:
        return "<0.001"

    return f"{p:.3f}"


# ============================================================
# 11. Dataset for statistical comparison
#
# IMPORTANT:
# Only IPV+TBI vs IPV alone are statistically compared.
# Overall IPV and CAPriCORN groups are descriptive only.
# ============================================================

test_df = df[
    df["cohort_label"].isin(
        ["IPV+TBI", "IPV alone"]
    )
].copy()

test_cohort_order = [
    "IPV+TBI",
    "IPV alone"
]


# ============================================================
# 12. AGE TEST
# Mann-Whitney U because age is reported as median (IQR)
# ============================================================

age_tbi = test_df.loc[
    test_df["cohort_label"] == "IPV+TBI",
    "age_at_index"
].dropna()

age_ipv = test_df.loc[
    test_df["cohort_label"] == "IPV alone",
    "age_at_index"
].dropna()

u_stat, age_p = mannwhitneyu(
    age_tbi,
    age_ipv,
    alternative="two-sided"
)


# ============================================================
# 13. Define categorical variables
# ============================================================

variables = [
    (
        "Sex",
        "SEX",
        [
            "Female",
            "Male",
            "Unknown"
        ]
    ),

    (
        "Race",
        "RACE",
        [
            "Asian",
            "African American",
            "White",
            "Other",
            "Unknown"
        ]
    ),

    (
        "Ethnicity",
        "HISPANIC",
        [
            "Non-Hispanic",
            "Hispanic",
            "Other",
            "Unknown"
        ]
    ),

    (
        "Sexual orientation",
        "SEXUAL_ORIENTATION",
        [
            "Straight",
            "LGBTQ+",
            "Other",
            "Unknown"
        ]
    ),

    (
        "Preferred language",
        "PAT_PREF_LANGUAGE_SPOKEN",
        [
            "English",
            "Spanish",
            "Other",
            "Unknown"
        ]
    ),

    (
        "Insurance",
        "payer_cat",
        [
            "Medicaid",
            "Medicare",
            "Private",
            "Other or unknown"
        ]
    )
]


# ============================================================
# 14. Run categorical statistical tests
#
# One overall test per VARIABLE.
#
# Chi-square is used by default.
#
# For a 2x2 table, Fisher's exact test is used if any
# expected cell count is <5.
# ============================================================

test_results = {}

for title, variable, levels in variables:

    tab = pd.crosstab(
        test_df["cohort_label"],
        test_df[variable]
    ).reindex(
        index=test_cohort_order,
        fill_value=0
    )

    # Remove categories that do not occur in either cohort
    tab = tab.loc[
        :,
        tab.sum(axis=0) > 0
    ]

    # Chi-square test
    chi2, p_chi, dof, expected = chi2_contingency(
        tab.values,
        correction=False
    )

    method = "Pearson chi-square"
    statistic = chi2
    p_value = p_chi
    result_df = dof

    # Fisher's exact test only for 2x2 tables
    # with small expected counts
    if (
        tab.shape == (2, 2)
        and (expected < 5).any()
    ):

        odds_ratio, p_fisher = fisher_exact(
            tab.values,
            alternative="two-sided"
        )

        method = "Fisher's exact"
        statistic = np.nan
        result_df = np.nan
        p_value = p_fisher

    test_results[title] = {
        "test": method,
        "statistic": statistic,
        "df": result_df,
        "p_value": p_value,
        "table": tab,
        "expected": expected
    }


# Add age result
test_results["Age"] = {
    "test": "Mann-Whitney U",
    "statistic": u_stat,
    "df": np.nan,
    "p_value": age_p
}


# ============================================================
# 15. Print statistical-test summary
# ============================================================

stat_rows = []

for variable in [
    "Age",
    "Sex",
    "Race",
    "Ethnicity",
    "Sexual orientation",
    "Preferred language",
    "Insurance"
]:

    result = test_results[variable]

    stat_rows.append({
        "Variable": variable,
        "Test": result["test"],
        "Statistic": result["statistic"],
        "df": result["df"],
        "p-value": p_fmt(
            result["p_value"]
        )
    })


statistical_results = pd.DataFrame(
    stat_rows
)

print(
    "\nStatistical comparison: "
    "IPV+TBI vs IPV alone"
)

display(statistical_results)


# ============================================================
# 16. BUILD TABLE 1
# ============================================================

rows = []


# ------------------------------------------------------------
# Age: median (IQR)
# P-value appears here
# ------------------------------------------------------------

row = {
    "Variable": "Age",
    "Label": "Median (IQR)",
    "P-value": p_fmt(age_p)
}

for cohort in cohort_order:

    x = df.loc[
        df["cohort_label"] == cohort,
        "age_at_index"
    ].dropna()

    if len(x) == 0:

        row[cohort] = ""

    else:

        row[cohort] = (
            f"{x.median():.0f} "
            f"({x.quantile(0.25):.0f}-"
            f"{x.quantile(0.75):.0f})"
        )

rows.append(row)


# ------------------------------------------------------------
# Age categories
#
# These are descriptive.
# No separate p-values because the age test above compares
# continuous age distributions.
# ------------------------------------------------------------

age_levels = [
    "18-34",
    "35-44",
    "45-54",
    "55-64",
    "65+"
]

for level in age_levels:

    row = {
        "Variable": "",
        "Label": level,
        "P-value": ""
    }

    for cohort in cohort_order:

        cohort_df = df[
            df["cohort_label"] == cohort
        ]

        denominator = len(
            cohort_df
        )

        n = (
            cohort_df["Age group"]
            .astype(str)
            .eq(level)
            .sum()
        )

        row[cohort] = format_n_pct(
            n,
            denominator
        )

    rows.append(row)


# ============================================================
# 17. Add categorical variables to Table 1
# ============================================================

for title, variable, levels in variables:

    first_row = True

    for level in levels:

        row = {
            "Variable":
                title if first_row else "",

            "Label":
                level,

            # ONE overall p-value for the entire variable
            "P-value":
                p_fmt(
                    test_results[title]["p_value"]
                )
                if first_row
                else ""
        }

        for cohort in cohort_order:

            cohort_df = df[
                df["cohort_label"] == cohort
            ]

            denominator = len(
                cohort_df
            )

            n = (
                cohort_df[variable]
                .eq(level)
                .sum()
            )

            row[cohort] = format_n_pct(
                n,
                denominator
            )

        rows.append(row)

        first_row = False


# ============================================================
# 18. Create final Table 1 dataframe
# ============================================================

table1 = pd.DataFrame(
    rows
)


# ============================================================
# 19. Add N to cohort column headers
# ============================================================

rename_columns = {}

for cohort in cohort_order:

    cohort_n = (
        df["cohort_label"]
        .eq(cohort)
        .sum()
    )

    rename_columns[cohort] = (
        f"{cohort}\n"
        f"N={cohort_n:,}"
    )

table1 = table1.rename(
    columns=rename_columns
)


# ============================================================
# 20. Put P-value at the end
# ============================================================

cohort_columns = [
    rename_columns[x]
    for x in cohort_order
]

table1 = table1[
    [
        "Variable",
        "Label"
    ]
    + cohort_columns
    + [
        "P-value"
    ]
]


# ============================================================
# 21. Display Table 1
# ============================================================

pd.set_option(
    "display.max_rows",
    200
)

print(
    "\nTABLE 1"
)

display(table1)


# ============================================================
# 22. Export Table 1 + statistical details to same Excel file
# ============================================================

with pd.ExcelWriter(
    "Table1_five_cohorts.xlsx"
) as writer:

    table1.to_excel(
        writer,
        sheet_name="Table 1",
        index=False
    )

    statistical_results.to_excel(
        writer,
        sheet_name="Statistical tests",
        index=False
    )


print(
    "\nSaved: Table1_five_cohorts.xlsx"
)

In [ ]:
# comorbidities 

import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, fisher_exact

def or_ci_woolf(a, b, c, d, correction_if_zero=True):
    if correction_if_zero and (a == 0 or b == 0 or c == 0 or d == 0):
        a, b, c, d = a + 0.5, b + 0.5, c + 0.5, d + 0.5

    or_val = (a * d) / (b * c)
    se = np.sqrt(1/a + 1/b + 1/c + 1/d)
    z = 1.959963984540054
    lo = np.exp(np.log(or_val) - z * se)
    hi = np.exp(np.log(or_val) + z * se)
    return or_val, lo, hi

def p_value_2x2(a, b, c, d):
    table = np.array([[a, b],
                      [c, d]])
    chi2, p_chi2, dof, expected = chi2_contingency(table, correction=False)

    if (expected < 5).any():
        _, p = fisher_exact(table)
        test = "Fisher"
    else:
        p = p_chi2
        test = "Chi-square"
    return p, test

def compute_comorbidity_stats(df, n_tbi, n_ipv):
    rows = []

    for _, r in df.iterrows():
        cat = r["category"]
        a = int(r["n_ipv_tbi"])
        c = int(r["n_ipv_alone"])
        b = n_tbi - a
        d = n_ipv - c

        p, test = p_value_2x2(a, b, c, d)
        or_val, lo, hi = or_ci_woolf(a, b, c, d)

        prev_tbi = a / n_tbi
        prev_ipv = c / n_ipv

        rows.append({
            "category": cat,
            "prev_ipv_tbi": prev_tbi,   # numeric for sorting
            "IPV+TBI": f"{a} ({prev_tbi*100:.1f}%)",
            "IPV only": f"{c} ({prev_ipv*100:.1f}%)",
            "p": p,
            "OR (95% CI)": f"{or_val:.2f} ({lo:.2f}–{hi:.2f})",
            "test": test
        })

    out = pd.DataFrame(rows)

    # sort by IPV+TBI prevalence descending
    out = out.sort_values("prev_ipv_tbi", ascending=False)

    # format p-values
    out["p-value"] = out["p"].apply(lambda x: "<0.001" if x < 0.001 else f"{x:.3f}")

    out = out.drop(columns=["p", "prev_ipv_tbi", "test"])
    return out.reset_index(drop=True)

data = [
    ("anxiety", 71, 311),
    ("hypertension", 74, 291),
    ("asthma", 58, 224),
    ("depression", 56, 252),
    ("drug_abuse", 32, 120),
    ("obesity", 45, 248),
    ("chronic_pain", 44, 190),
    ("hyperlipidemia", 37, 158),
    ("suicidal_homicidal_ideation_or_attempt", 24, 74),
    ("alcohol_abuse", 28, 60),
    ("diabetes_without_complication", 23, 119),
    ("deficiency_anemia", 29, 70),
    ("PTSD", 18, 67),
    ("cerebrovascular_disease_and_stroke", 21, 44),
    ("chronic_liver_disease_and_cirrhosis", 23, 55),
    ("coronary_ischemic_heart_disease", 19, 86),
    ("peripheral_vascular_disease", 16, 43),
    ("schizophrenia_and_psychotic_disorders", 11, 59),
    ("diabetes_with_complication", 16, 78),
    ("COPD", 15, 46),
    ("congestive_heart_failure", 13, 57),
    ("chronic_kidney_disease", 11, 54),
    ("hiv_aids", 3, 11),
    ("all_cancer", 6, 31),
]
df_counts = pd.DataFrame(data, columns=["category", "n_ipv_tbi", "n_ipv_alone"])

result = compute_comorbidity_stats(df_counts, n_tbi=486, n_ipv=2330)

result
# save result to csv
#result.to_csv("comorbidity_comparison.csv", index=False)

In [ ]:
# healthcare utilization 

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency, fisher_exact


# ============================================================
# 1. Load and clean data
# ============================================================

hlth_util = pd.read_csv("healthcare_util.csv")

hlth_util["cohort_label"] = (
    hlth_util["cohort_label"]
    .astype(str)
    .str.strip()
    .replace({"IPV-alone": "IPV alone"})
)

cohort_order = ["IPV+TBI", "IPV alone"]


# Confirm that the expected cohort labels exist
print("Cohort counts:")
print(hlth_util["cohort_label"].value_counts(dropna=False))


# ============================================================
# 2. Validate binary variables
# ============================================================

binary_variables = ["any_ambulatory", "any_acute"]


binary_mapping = {
    "0": 0,
    "1": 1,
    "false": 0,
    "true": 1,
    "no": 0,
    "yes": 1,
    "n": 0,
    "y": 1,
}

for var in binary_variables:
    raw_values = hlth_util[var].copy()

    converted = (
        raw_values
        .astype(str)
        .str.strip()
        .str.lower()
        .map(binary_mapping)
    )

    if converted.isna().any():
        invalid_values = raw_values[converted.isna()].unique()

        raise ValueError(
            f"Column '{var}' contains missing or unrecognized values: "
            f"{invalid_values}"
        )

    hlth_util[var] = converted.astype(int)


# ============================================================
# 3. Patient-level utilization summary
# ============================================================

summary = (
    hlth_util.groupby("cohort_label")
    .agg(
        cohort_n=("PATID", "count"),

        n_any_ambulatory=("any_ambulatory", "sum"),
        pct_any_ambulatory=(
            "any_ambulatory",
            lambda s: 100 * s.mean()
        ),

        n_any_acute=("any_acute", "sum"),
        pct_any_acute=(
            "any_acute",
            lambda s: 100 * s.mean()
        ),
    )
    .reindex(cohort_order)
)


# Confirm neither cohort was accidentally lost
if summary["cohort_n"].isna().any():
    missing_cohorts = summary.index[
        summary["cohort_n"].isna()
    ].tolist()

    raise ValueError(
        f"No data found for cohort(s): {missing_cohorts}. "
        f"Available labels are: "
        f"{hlth_util['cohort_label'].unique().tolist()}"
    )


# ============================================================
# 4. Formatting and Holm adjustment functions
# ============================================================

def p_fmt(p):
    """Format p-values for tables and figures."""
    if p < 0.001:
        return "<0.001"

    return f"{p:.3f}"


def holm_adjust(pvals):
    """
    Holm-Bonferroni adjustment.

    Returns adjusted p-values in the same order as the inputs.
    """
    pvals = np.asarray(pvals, dtype=float)
    m = len(pvals)

    order = np.argsort(pvals)
    adjusted = np.empty(m, dtype=float)

    running_max = 0.0

    for rank, idx in enumerate(order):
        adjusted_value = (m - rank) * pvals[idx]
        running_max = max(running_max, adjusted_value)
        adjusted[idx] = min(running_max, 1.0)

    return adjusted


# ============================================================
# 5. Chi-square or Fisher's exact tests
# ============================================================

test_results = {}

for var in binary_variables:

    contingency_table = (
        pd.crosstab(
            hlth_util["cohort_label"],
            hlth_util[var]
        )
        .reindex(
            index=cohort_order,
            columns=[0, 1],
            fill_value=0
        )
    )

    print(f"\nContingency table for {var}:")
    print(contingency_table)

    # Confirm both cohorts contain records
    if (contingency_table.sum(axis=1) == 0).any():
        missing_cohorts = contingency_table.index[
            contingency_table.sum(axis=1) == 0
        ].tolist()

        raise ValueError(
            f"No observations were found for cohort(s): "
            f"{missing_cohorts}"
        )

    # Confirm the outcome contains both 0 and 1 values
    if (contingency_table.sum(axis=0) == 0).any():
        missing_categories = contingency_table.columns[
            contingency_table.sum(axis=0) == 0
        ].tolist()

        raise ValueError(
            f"Outcome '{var}' does not contain response "
            f"category/categories {missing_categories}."
        )

    chi2, p_chi, dof, expected = chi2_contingency(
        contingency_table.values,
        correction=False
    )

    if (expected < 5).any():
        odds_ratio, p_value = fisher_exact(
            contingency_table.values,
            alternative="two-sided"
        )

        method = "Fisher's exact test"
        test_statistic = np.nan

    else:
        p_value = p_chi
        method = "Pearson chi-square test"
        test_statistic = chi2

    test_results[var] = {
        "method": method,
        "p_raw": p_value,
        "chi_square": test_statistic,
        "degrees_freedom": dof,
        "table": contingency_table,
        "expected": expected,
    }


# ============================================================
# 6. Holm correction across the two utilization comparisons
# ============================================================

test_keys = [
    "any_ambulatory",
    "any_acute",
]

adjusted_p_values = holm_adjust(
    [test_results[key]["p_raw"] for key in test_keys]
)

for key, adjusted_p in zip(test_keys, adjusted_p_values):
    test_results[key]["p_holm"] = adjusted_p


# ============================================================
# 7. Display statistical-test results
# ============================================================

test_table = pd.DataFrame(
    [
        {
            "Outcome": "Any ambulatory visit",
            "Test": test_results["any_ambulatory"]["method"],
            "Raw p-value": test_results["any_ambulatory"]["p_raw"],
            "Holm-adjusted p-value":
                test_results["any_ambulatory"]["p_holm"],
        },
        {
            "Outcome": "Any acute care visit",
            "Test": test_results["any_acute"]["method"],
            "Raw p-value": test_results["any_acute"]["p_raw"],
            "Holm-adjusted p-value":
                test_results["any_acute"]["p_holm"],
        },
    ]
)

display(
    test_table.assign(
        **{
            "Raw p-value":
                test_table["Raw p-value"].map(p_fmt),

            "Holm-adjusted p-value":
                test_table["Holm-adjusted p-value"].map(p_fmt),
        }
    )
)


# ============================================================
# 8. Create the two-panel figure
# ============================================================

sns.set(
    style="whitegrid",
    context="talk"
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(11, 5.5),
    constrained_layout=True
)

palette = {
    "IPV+TBI": "#d95f02",
    "IPV alone": "#1b9e77",
}


# ------------------------------------------------------------
# Panel (a): Any ambulatory utilization
# ------------------------------------------------------------

sns.barplot(
    x=summary.index,
    y=summary["pct_any_ambulatory"].values,
    order=cohort_order,
    palette=palette,
    ax=axes[0]
)

axes[0].set_title("Any Ambulatory Visits")
axes[0].set_xlabel("")
axes[0].set_ylabel("Patients (%)")

ambulatory_ymax = max(
    60,
    summary["pct_any_ambulatory"].max() * 1.35
)

axes[0].set_ylim(0, ambulatory_ymax)

for i, cohort in enumerate(cohort_order):

    percentage = summary.loc[
        cohort,
        "pct_any_ambulatory"
    ]

    patient_count = int(
        summary.loc[
            cohort,
            "n_any_ambulatory"
        ]
    )

    axes[0].text(
        i,
        percentage + 1,
        f"{percentage:.1f}%\n(N={patient_count})",
        ha="center",
        va="bottom",
        fontsize=11
    )

axes[0].text(
    0.98,
    0.98,
    f"{test_results['any_ambulatory']['method']}\n"
    f"p = {p_fmt(test_results['any_ambulatory']['p_raw'])}\n"
    f"Holm p = "
    f"{p_fmt(test_results['any_ambulatory']['p_holm'])}",
    transform=axes[0].transAxes,
    ha="right",
    va="top",
    fontsize=9.5,
    bbox={
        "boxstyle": "round,pad=0.25",
        "facecolor": "white",
        "edgecolor": "gray",
        "alpha": 0.9,
    }
)


# ------------------------------------------------------------
# Panel (b): Any acute care utilization
# ------------------------------------------------------------

sns.barplot(
    x=summary.index,
    y=summary["pct_any_acute"].values,
    order=cohort_order,
    palette=palette,
    ax=axes[1]
)

axes[1].set_title("Any Acute Care Visits")
axes[1].set_xlabel("")
axes[1].set_ylabel("Patients (%)")

acute_ymax = max(
    60,
    summary["pct_any_acute"].max() * 1.35
)

axes[1].set_ylim(0, acute_ymax)

for i, cohort in enumerate(cohort_order):

    percentage = summary.loc[
        cohort,
        "pct_any_acute"
    ]

    patient_count = int(
        summary.loc[
            cohort,
            "n_any_acute"
        ]
    )

    axes[1].text(
        i,
        percentage + 1,
        f"{percentage:.1f}%\n(N={patient_count})",
        ha="center",
        va="bottom",
        fontsize=11
    )

axes[1].text(
    0.98,
    0.98,
    f"{test_results['any_acute']['method']}\n"
    f"p = {p_fmt(test_results['any_acute']['p_raw'])}\n"
    f"Holm p = "
    f"{p_fmt(test_results['any_acute']['p_holm'])}",
    transform=axes[1].transAxes,
    ha="right",
    va="top",
    fontsize=9.5,
    bbox={
        "boxstyle": "round,pad=0.25",
        "facecolor": "white",
        "edgecolor": "gray",
        "alpha": 0.9,
    }
)


# ============================================================
# 9. Panel labels and figure title
# ============================================================

for ax, panel_label in zip(
    axes,
    ["(a)", "(b)"]
):
    ax.text(
        -0.12,
        1.06,
        panel_label,
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=14,
        fontweight="bold"
    )

fig.suptitle(
    "Pre-index Healthcare Utilization Patterns by Cohort",
    fontsize=18
)

plt.show()


# ============================================================
# 10. Display summary data
# ============================================================

display(summary.round(2))